# AGE & GENDER PREDICTOR




### **__Objective__**: 
The goal of this notebook is to develop and train a deep learning model to predict a person's age and gender from a facial image. This is a classic multi-task learning problem, as our single model must perform two different tasks simultaneously:

- Age Prediction: A regression task to predict a continuous numeric value.
- Gender Prediction: A binary classification task to predict a 0 or 1.


### **__Technologies and Approach__**:
- Core Framework: PyTorch & PyTorch Lightning

- Key Libraries: Pandas, NumPy, PIL, Torchvision, TorchMetrics

- Experiment Tracking: TrackIO (logging to Hugging Face)

- Model Architecture: EfficientNetV2-S (pre-trained)

- Loss Functions:

> a) nn.SmoothL1Loss (for Age)
> 
> b) nn.BCEWithLogitsLoss (for Gender)

- Optimization:

> AdamW Optimizer
> 
> CosineAnnealingLR (Scheduler)

- Training & Regularization:

> Multi-Task Weighted Loss (loss_weight_age=0.34)
> 
> Data Augmentation (Flip, Jitter, Affine, RandomErasing)
>
> Dropout
>
> Stochastic Weight Averaging (SWA)

- Inference:

> Test-Time Augmentation (TTA)

# 1. Setup and Configuration

In [1]:
%%capture
%pip install trackio -qq
%pip uninstall -y protobuf -qq
%pip install protobuf==3.20.1 -qq

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
import torchvision.models as models

import warnings
warnings.filterwarnings('ignore')

# for logging
import trackio

import kagglehub
from pathlib import Path

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, Callback, StochasticWeightAveraging
import torchmetrics

In [3]:
torch.cuda.empty_cache()

In [4]:
# Huggingface Access Adding token to environment variable
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

os.environ['HF_TOKEN']  = user_secrets.get_secret("HF_TOKEN")

In [5]:
# ENV Paths
DATA_DIR = "/kaggle/input/sep-25-dl-gen-ai-nppe-1/face_dataset/" 
TRAIN_CSV_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_CSV_PATH = os.path.join(DATA_DIR, "test.csv")
TRAIN_IMG_DIR = DATA_DIR
TEST_IMG_DIR = DATA_DIR

# Trackio routes 
TRACKIO_SPACE_ID = "22f3001059/dlgenai-nppe" 
TRACKIO_PROJECT = "25-t3-nppe1"

# Model & Training Hyperparameters
IMG_SIZE = 300
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
MAX_EPOCHS = 31
NUM_WORKERS = os.cpu_count()
VAL_SPLIT = 0.2 

pl.seed_everything(42)

Seed set to 42


42

# 2. PyTorch Dataset

In [6]:
# Custom Dataset for loading face images and their labels (age, gender).
# For the test set, it will only load images.

class FaceDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None, is_test=False):
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        # 'full_path' column has relative path, joining it with base image dir 
        img_path = os.path.join(self.img_dir, self.data.iloc[idx]['full_path'])

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            # for test set, only return the image and its ID
            image_id = self.data.iloc[idx]['id']
            return image, image_id
        else:
            # for train set, return image and labels
            age = self.data.iloc[idx]['age']
            gender = self.data.iloc[idx]['gender']
            
            # Convert labels to appropriate torch tensors
            age = torch.tensor(age, dtype=torch.float32)
            gender = torch.tensor(gender, dtype=torch.float32) # For BCEWithLogitsLoss

            return image, {"age": age, "gender": gender}

# 3. PyTorch Lightning DataModule

In [7]:
# LightningDataModule to handle all data loading and splitting.

class FaceDataModule(pl.LightningDataModule):
    def __init__(self, train_csv, test_csv, train_img_dir, test_img_dir, batch_size, val_split, num_workers):
        super().__init__()
        self.train_csv = train_csv
        self.test_csv = test_csv
        self.train_img_dir = train_img_dir
        self.test_img_dir = test_img_dir
        self.batch_size = batch_size
        self.val_split = val_split
        self.num_workers = num_workers

        # define transformations
        
        self.train_transform = T.Compose([
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.RandomHorizontalFlip(),
            T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
            T.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225]),
            T.RandomErasing(p=0.5, scale=(0.02, 0.1), ratio=(0.3, 3.3)),
        ])

        self.val_test_transform = T.Compose([
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225]),
        ])
        
        # for test-time augmentation
        self.tta_transform = T.Compose([
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.RandomHorizontalFlip(p=1.0), # always flip
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225]),
        ])

    def setup(self, stage=None): 
        if stage == 'fit' or stage is None:
            full_dataset = FaceDataset(self.train_csv, self.train_img_dir, transform=self.train_transform) # load training dataset
            
            # split into train and validation
            total_len = len(full_dataset)
            val_len = int(total_len * self.val_split)
            train_len = total_len - val_len
            
            self.train_dataset, self.val_dataset = random_split(full_dataset, [train_len, val_len])
            
            # treat val_dataset like test dataset, hence no augmentation on that
            self.val_dataset.dataset.transform = self.val_test_transform

        if stage == 'predict' or stage is None:
            self.test_dataset = FaceDataset(self.test_csv, self.test_img_dir, transform=self.val_test_transform, is_test=True)
            self.test_dataset_tta = FaceDataset(self.test_csv, self.test_img_dir,transform=self.tta_transform, is_test=True)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, pin_memory=True)

    def predict_dataloader(self):
        return [DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, pin_memory=True),
                DataLoader(self.test_dataset_tta, batch_size=self.batch_size,shuffle=False, num_workers=self.num_workers, pin_memory=True)]

# 4. PyTorch Lightning Model

## a) **Defining a Basic CNN Model from Scratch**

In [8]:
# # a from scratch implementation 

# class ScratchCNN(pl.LightningModule):
#     def __init__(self, learning_rate=1e-4, loss_weight_age=0.1):
#         super().__init__()
#         self.save_hyperparameters()
        
#         self.learning_rate = learning_rate
#         self.loss_weight_age = loss_weight_age

#         # making a backbone, then we can attack any head to it that we want to 
        
#         self.backbone = nn.Sequential(
#             # Input: (B, 3, H, W)
            
#             nn.Conv2d(3, 16, kernel_size=3, padding=1, stride=1), # -> (B, 16, H, W)
#             nn.ReLU(),
#             nn.BatchNorm2d(16),
#             nn.MaxPool2d(2, 2), # -> (B, 16, H/2, W/2)
            
#             nn.Conv2d(16, 32, kernel_size=3, padding=1, stride=1), # -> (B, 32, H/2, W/2)
#             nn.ReLU(),
#             nn.BatchNorm2d(32),
#             nn.MaxPool2d(2, 2), # -> (B, 32, H/4, W/4)
            
#             nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=1), # -> (B, 64, H/4, W/4)
#             nn.ReLU(),
#             nn.BatchNorm2d(64),
#             nn.MaxPool2d(2, 2), # -> (B, 64, H/8, W/8)
            
#             # global pooling
#             nn.AdaptiveAvgPool2d((1, 1)), # -> (B, 64, 1, 1)

#             # flattening for fc layer
#             nn.Flatten() # -> (B, 64)
#         )
        
#         # number of features is the output of the last conv block (64)
#         num_features = 64
        
        
#         # now we create 2 seperate heads for our 2 tasks respectively - age (regression) and gender (classification)
#         # age head (regression)
#         self.age_head = nn.Sequential(
#             nn.Linear(num_features, 128),
#             nn.ReLU(),
#             nn.Linear(128, 1)  
#         )
        
#         # gender head (binary classification)
#         self.gender_head = nn.Sequential(
#             nn.Linear(num_features, 128),
#             nn.ReLU(),
#             nn.Linear(128, 1)  
#         )
        
#         # defining loss functions - MSE for age (regression) and BCE for gender (classification)
#         # Using SmoothL1Loss for age regression is a great choice as it's less sensitive to outliers than MSE
#         self.age_loss_fn = nn.SmoothL1Loss()  
#         self.gender_loss_fn = nn.BCEWithLogitsLoss()
        
#         # eval metrics (as decided by the comepeition) - RMSE for age, F1 for gender
#         self.val_age_rmse = torchmetrics.regression.MeanSquaredError(squared=False)
#         self.val_gender_f1 = torchmetrics.classification.BinaryF1Score()
        
#         self.train_age_rmse = torchmetrics.regression.MeanSquaredError(squared=False)
#         self.train_gender_f1 = torchmetrics.classification.BinaryF1Score()

    
#     def forward(self, x):
#         # x is the batch of images
#         features = self.backbone(x) # (B, 64)
        
#         # Pass features through each head
#         age_pred = self.age_head(features).squeeze(-1) # removes last dimension
#         gender_pred = self.gender_head(features).squeeze(-1) # removes last dimension 
        
#         return {"age": age_pred, "gender": gender_pred} 

        
#     def training_step(self, batch, batch_idx):
        
#         images, age_labels, gender_labels = batch
        
#         age_pred_logits, gender_pred_logits = self(images)
        
#         age_pred = age_pred_logits.squeeze() # (B,)
#         gender_pred = gender_pred_logits.squeeze() # (B,)
        

#         age_labels = age_labels.float()
#         gender_labels = gender_labels.float()

#         age_loss = self.age_loss_fn(age_pred, age_labels)
#         gender_loss = self.gender_loss_fn(gender_pred, gender_labels)
        
#         # weighted loss
#         total_loss = (self.loss_weight_age * age_loss) + (1.0 - self.loss_weight_age) * gender_loss
        
#         self.train_age_rmse(age_pred, age_labels)
#         self.train_gender_f1(torch.sigmoid(gender_pred), gender_labels.int())
        
#         self.log("train_loss", total_loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
#         self.log("train_age_loss", loss_age, on_step=False, on_epoch=True, logger=True)
#         self.log("train_gender_loss", loss_gender, on_step=False, on_epoch=True, logger=True)
#         self.log("train_age_rmse", self.train_age_rmse, on_step=False, on_epoch=True, prog_bar=True, logger=True)
#         self.log("train_f1", self.train_gender_f1, on_step=False, on_epoch=True, prog_bar=True, logger=True)

#         if trackio:
#             try:
#                 trackio.log({
#                     "train_loss": total_loss.item(),
#                     "train_age_loss": loss_age.item(),
#                     "train_gender_loss": loss_gender.item()
#                 })
#             except Exception as e:
#                 print(f"Trackio step log failed: {e}")
        
#         return total_loss
        

#     def validation_step(self, batch, batch_idx):
#         images, age_labels, gender_labels = batch
        
#         age_pred_logits, gender_pred_logits = self(images)
        
#         age_pred = age_pred_logits.squeeze()
#         gender_pred = gender_pred_logits.squeeze()

#         age_labels = age_labels.float()
#         gender_labels = gender_labels.float()
        
#         age_loss = self.age_loss_fn(age_pred, age_labels)
#         gender_loss = self.gender_loss_fn(gender_pred, gender_labels)
        
#         total_loss = (self.loss_weight_age * age_loss) + (1.0 - self.loss_weight_age) * gender_loss # weighted loss
        
#         # Update validation metrics
#         self.val_age_rmse(age_pred, age_labels)
#         self.val_gender_f1(torch.sigmoid(gender_pred), gender_labels.int())
        
#         # Log metrics
#         self.log('val/loss_total', total_loss, on_step=False, on_epoch=True, prog_bar=True)
#         self.log('val/loss_age', age_loss, on_step=False, on_epoch=True)
#         self.log('val/loss_gender', gender_loss, on_step=False, on_epoch=True)
#         self.log('val/rmse_age', self.val_age_rmse, on_step=False, on_epoch=True, prog_bar=True)
#         self.log('val/f1_gender', self.val_gender_f1, on_step=False, on_epoch=True, prog_bar=True)

#         if trackio:
#             try:
#                 trackio.log({
#                     "val_loss": total_loss.item(),
#                     "val_rmse": loss_age.item(),
#                     "val_f1": loss_gender.item()
#                 })
#             except Exception as e:
#                 print(f"Trackio step log failed: {e}")


#     def configure_optimizers(self):
#         optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
#         return optimizer

## b) Fine-tuning a Pretrained Model

In [9]:
# LightningModule for Age (Regression) and Gender (Classification).
# Uses a pre-trained ResNet18 backbone.

class AgeGenderModel(pl.LightningModule):
    def __init__(self, learning_rate=1e-4, loss_weight_age=0.1):
            super().__init__()
            self.save_hyperparameters()
        
            self.learning_rate = learning_rate

            self.loss_weight_age = loss_weight_age
    
            self.backbone = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
            
            num_features = self.backbone.classifier[1].in_features # get the number of features from the layer before the final classifier
            
            self.backbone.classifier[1] = nn.Identity() # remove the original classifier
    
            # now we create 2 seperate heads for our 2 tasks respectively - age (regression) and gender (classification)
            # age head (regression)
            self.age_head = nn.Sequential(
                nn.Linear(num_features, 128),
                nn.ReLU(),
                nn.Dropout(p=0.4),
                nn.Linear(128, 1) 
            )
            
            # gender head (binary classification)
            self.gender_head = nn.Sequential(
                nn.Linear(num_features, 128),
                nn.ReLU(),
                nn.Dropout(p=0.4),
                nn.Linear(128, 1) 
            )
    
            # defining loss functions - MSE for age (regression) and BCE for gender (classification)

            self.age_loss_fn = nn.SmoothL1Loss() 
            self.gender_loss_fn = nn.BCEWithLogitsLoss()
    
            # eval metrics (as decided by the comepeition) - RMSE for age, F1 for gender

            self.val_age_rmse = torchmetrics.regression.MeanSquaredError(squared=False)
            self.val_gender_f1 = torchmetrics.classification.BinaryF1Score()
            
            self.train_age_rmse = torchmetrics.regression.MeanSquaredError(squared=False)
            self.train_gender_f1 = torchmetrics.classification.BinaryF1Score()


    def forward(self, x):
        # pass input through the backbone, aand then pass features through each head
        features = self.backbone(x)
        
        age_pred = self.age_head(features).squeeze(-1) # removes last dimension
        
        gender_pred = self.gender_head(features).squeeze(-1) # removes last dimension 
        
        return {"age": age_pred, "gender": gender_pred}


    def training_step(self, batch, batch_idx):
        images, targets = batch

        # take predictions
        preds = self(images)
        
        age_target = targets["age"]
        gender_target = targets["gender"]
        
        age_pred = preds["age"]
        gender_pred = preds["gender"]

        # calculate losses
        loss_age = self.age_loss_fn(age_pred, age_target)
        loss_gender = self.gender_loss_fn(gender_pred, gender_target)
        
        # weighted loss
        total_loss = (self.loss_weight_age * loss_age) + loss_gender

        # los metrics
        self.train_age_rmse(age_pred, age_target)
        self.train_gender_f1(gender_pred, gender_target)
        
        self.log("train_loss", total_loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        self.log("train_age_loss", loss_age, on_step=False, on_epoch=True, logger=True)
        self.log("train_gender_loss", loss_gender, on_step=False, on_epoch=True, logger=True)
        self.log("train_age_rmse", self.train_age_rmse, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        self.log("train_f1", self.train_gender_f1, on_step=False, on_epoch=True, prog_bar=True, logger=True)

        if trackio:
            try:
                trackio.log({
                    "train_loss": total_loss.item(),
                    "train_age_loss": loss_age.item(),
                    "train_gender_loss": loss_gender.item()
                })
            except Exception as e:
                print(f"Trackio step log failed: {e}")
        
        return total_loss
        
        
    # def on_validation_epoch_end(self):
    #     if not self.trainer.sanity_checking and trackio:
    #         try:
    #             metrics = self.trainer.logged_metrics
    #             trackio.log({
    #                 "epoch": float(self.current_epoch),
    #                 "val_loss": metrics.get("val_loss", 0.0).item(),
    #                 "val_rmse": metrics.get("val_rmse", 0.0).item(),
    #                 "val_f1": metrics.get("val_f1", 0.0).item(),
    #                 # Also log training epoch metrics
    #                 "train_loss_epoch": metrics.get("total_loss", 0.0).item(),
    #                 "train_rmse_epoch": metrics.get("train_rmse", 0.0).item(),
    #                 "train_f1_epoch": metrics.get("train_f1", 0.0).item(),
    #             })
    #             print(f"Logged epoch {self.current_epoch} metrics to Trackio.")
    #         except Exception as e:
    #             print(f"Trackio epoch log failed: {e}")
    
    def validation_step(self, batch, batch_idx):
        images, targets = batch
        
        preds = self(images)
        
        age_target = targets["age"]
        gender_target = targets["gender"]
        
        age_pred = preds["age"]
        gender_pred = preds["gender"]

        loss_age = self.age_loss_fn(age_pred, age_target)
        loss_gender = self.gender_loss_fn(gender_pred, gender_target)
        
        total_loss = (self.loss_weight_age * loss_age) + loss_gender

        self.val_age_rmse(age_pred, age_target)
        self.val_gender_f1(gender_pred, gender_target)

        self.log("val_loss", total_loss, on_epoch=True, prog_bar=True, logger=True)
        self.log("val_rmse", self.val_age_rmse, on_epoch=True, prog_bar=True, logger=True)
        self.log("val_f1", self.val_gender_f1, on_epoch=True, prog_bar=True, logger=True)

        if trackio:
            try:
                trackio.log({
                    "val_loss": total_loss.item(),
                    "val_rmse": loss_age.item(),
                    "val_f1": loss_gender.item()
                })
            except Exception as e:
                print(f"Trackio step log failed: {e}")

    
    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        # preds on test set
        images, image_ids = batch
        
        preds = self(images)
        
        # age_pred = preds["age"].clamp(min=0) # keeping age as float, also making sure age > 0
        age_pred = preds["age"]
        
        # gender_pred = (preds["gender"] > 0).int() # use logit to save computation, instead of using sigmoid > 0.5
        gender_pred = preds["gender"]
        
        return image_ids, age_pred, gender_pred

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate)
        
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)
        
        return {
            "optimizer": optimizer, 
            "lr_scheduler": {
                "scheduler":scheduler, 
                "interval":"epoch",
                "monitor":"val_rmse"
            }
        }

# 5. Initialize Model & Trainer

In [10]:
data_module = FaceDataModule(
    train_csv=TRAIN_CSV_PATH,
    test_csv=TEST_CSV_PATH,
    train_img_dir=TRAIN_IMG_DIR, 
    test_img_dir=TEST_IMG_DIR,  
    batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT,
    num_workers=NUM_WORKERS
)

print("Initializing Model...")

model = AgeGenderModel(learning_rate=LEARNING_RATE, loss_weight_age=0.34)

Initializing Model...


Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth
100%|██████████| 82.7M/82.7M [00:00<00:00, 161MB/s]


In [11]:
#  setting up callbacks, and save the best model based on validation loss
checkpoint_callback = ModelCheckpoint(
    monitor="val_rmse",
    dirpath="checkpoints",
    filename="best-model-{epoch:02d}-{val_loss:.2f}",
    save_top_k=1,
    mode="min",
)

In [12]:
swa_callback = StochasticWeightAveraging(swa_lrs=1e-2)

In [13]:
# initialize trainer
trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="auto", # Uses GPU if available
    # devices=2,
    # precision="16-mixed",
    callbacks=[checkpoint_callback, swa_callback], # Add trackio callback
    logger=True, # Uses TensorBoardLogger by default
    log_every_n_steps=10,
    # precision=16
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


# 6. Training and Run

In [14]:
# # initiazlise trackio run 
# trackio.init(
#     space_id=TRACKIO_SPACE_ID,
#     project=TRACKIO_PROJECT,
#     config={"epochs": MAX_EPOCHS, "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE},
#     group="efficientnet_v2_s_",  
#     name="run_5"               
# )
# print(f"Trackio initialized for project: {TRACKIO_PROJECT}")

# print("Starting model training...")

# try:
#     trainer.fit(model, data_module)
#     print("Training complete.")
# except Exception as e:
#     print(f"error:{e}")


# if trackio:
#     try:
#         trackio.finish()
#         print("Trackio run finished.")
#     except Exception as e:
#         print(f"error occured while trackio run: {e}")

# model_path = Path(trainer.checkpoint_callback.best_model_path).parent

# KAGGLE_USERNAME = 'gaurangnigam'
# MODEL = 'dlgenai-nppe' # different architectures
# FRAMEWORK = 'pytorch'
# VARIATION = 'efficientnetv2s'
# handle = f'{KAGGLE_USERNAME}/{MODEL}/{FRAMEWORK}/{VARIATION}'
# kagglehub.model_upload(handle, model_path, version_notes='initial model')

# 7. Creating submission file 

In [15]:
# model_path = '/kaggle/input/dlegenai-nppe/pytorch/efficientnetv2s/model.ckpt'
model_path = '/kaggle/input/dlgenai-nppe/pytorch/efficientnetv2s/5/best-model-epoch18-val_loss1.85.ckpt'

model =  AgeGenderModel.load_from_checkpoint(model_path)

In [16]:
predictions_list = trainer.predict(model, data_module)

preds_orig = predictions_list[0]
preds_tta = predictions_list[1]

# store all predictions in dictionaries keyed by image_id
# we average age (float) and gender (logits)
agg_preds = {}

# process original predictions
for batch_preds in preds_orig:
    image_ids, age_preds, gender_preds = batch_preds
    for i in range(len(image_ids)):
        img_id = image_ids[i]
        agg_preds[img_id] = {
            "age": [age_preds[i].item()],
            "gender_logit": [gender_preds[i].item()]
        }

# process flipped (TTA) predictions
for batch_preds in preds_tta:
    image_ids, age_preds, gender_preds = batch_preds
    for i in range(len(image_ids)):
        img_id = image_ids[i]
        if img_id in agg_preds:
            agg_preds[img_id]["age"].append(age_preds[i].item())
            agg_preds[img_id]["gender_logit"].append(gender_preds[i].item())

submission = []

for img_id, preds in agg_preds.items():
    avg_age = np.mean(preds["age"])
    avg_gender_logit = np.mean(preds["gender_logit"])
    
    final_age = max(0.0, avg_age) # ReLU(age) haha, basically checking if age > 0
    final_gender = 1 if avg_gender_logit > 0 else 0 # Convert avg logit to 0 or 1
    
    submission.append({
        "id": img_id,
        "age": final_age,
        "gender": final_gender
    })

submission_df = pd.DataFrame(submission)
submission_df['age'] = submission_df['age'].astype(float)
submission_df['gender'] = submission_df['gender'].astype(int)
submission_df = submission_df.sort_values(by='id').set_index('id')

sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
submission_df = submission_df.reindex(sample_sub['id'].values)
submission_df = submission_df.reset_index()


submission_df.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")

submission_df.head()

2025-11-10 18:34:34.075794: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762799674.257019      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762799674.303506      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

submission.csv created successfully!


,id,age,gender
0,0,33.679016,1
1,1,26.774122,0
2,2,25.269754,1
3,3,35.486073,1
4,4,58.734398,1
